In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import concurrent.futures
import functools

from func_file_Process_exp import *

In [ ]:
#Create folder for storing generated data files
if not os.path.exists("Data_files"):
    os.mkdir("Data_files")
if not os.path.exists("Data_files/Processed_data"):
    os.mkdir("Data_files/Processed_data")

## Load the ConvNet model and Specify parameters for Richardson-Lucy algorithms

In [ ]:
#Load the ConvNet model
model_ConvNet = Load_ConvNet_model("DAMN_ConvNet.h5")

In [ ]:
#Specify the PSF_width value of the experiment; requirement for Richardson-Lucy algorithm
PSF_width_value = 2.05
kernel = Airy_kernel(PSF_width_value)      #The PSF of the experiment is Airy

#To fasten the Richardson-Lucy algorithm, we split the data evaluation among several CPU units using ProcessPoolExecutor from concurrent.futures
CPU_units_to_use = 4

## Evaluate the experiment dataset

## 1) Simulated Data

In [ ]:
#Loading the data generated in the Data_generation notebook
data_in_sim = np.load("Data_files/Generated_data/Simulated_data_low_res.npy")
data_target_sim = np.load("Data_files/Generated_data/Simulated_data_high_res.npy")

shape_sim = data_in_sim.shape
print("Data shapes:", shape_sim)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_sim = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_sim:
    RL_data_in_sim = data_in_sim
    RL_data_target_sim = data_target_sim
    
    RL_output_sim = np.zeros(data_in_sim.shape)
    RL_TV_output_sim = np.zeros(data_in_sim.shape)
    
else:
    RL_data_in_sim = data_in_sim[::4,:10]
    RL_data_target_sim = data_target_sim[::4,:10]
    
    RL_output_sim = np.zeros(data_in_sim[::4,:10].shape)
    RL_TV_output_sim = np.zeros(data_in_sim[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_sim.shape)

### The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_sim = np.reshape(model_ConvNet.predict(data_in_sim.reshape([shape_sim[0]*shape_sim[1], shape_sim[2], shape_sim[3]]), 
                                                         verbose = 1), shape_sim)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_sim = Evaluate_metric(DAMN_model_output_sim, data_target_sim)

np.save("Data_files/Processed_data/Sim_DAMN_model_output.npy", DAMN_model_output_sim)
np.save("Data_files/Processed_data/Sim_DAMN_model_errors.npy", DAMN_model_errors_sim)

### Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the simulated data
start = time.time()
for i in range(RL_data_in_sim.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_sim[i])
    RL_output_sim[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_sim.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Sim_RL_output.npy", RL_output_sim)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_sim = Evaluate_metric(RL_output_sim, RL_data_target_sim)

np.save("Data_files/Processed_data/Sim_RL_errors.npy", RL_errors_sim)

### Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the simulated data
start = time.time()
for i in range(RL_data_in_sim.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_sim[i])
    RL_TV_output_sim[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_sim.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Sim_RLTV_output.npy", RL_TV_output_sim)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_sim = Evaluate_metric(RL_TV_output_sim, RL_data_target_sim)

np.save("Data_files/Processed_data/Sim_RLTV_errors.npy", RL_TV_errors_sim)

## 2) Measured Data

In [ ]:
#Loading the data generated in the Data_generation notebook
datafile = np.load("Data_files/Measured_data.npz")
data_in_exp = datafile["LR_meas_data"]
data_target_exp = datafile["LR_meas_data"]

shape_exp = data_in_exp.shape
print("Data shapes:", shape_exp)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_exp = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_exp:
    RL_data_in_exp = data_in_exp
    RL_data_target_exp = data_target_exp
    
    RL_output_exp = np.zeros(data_in_exp.shape)
    RL_TV_output_exp = np.zeros(data_in_exp.shape)
    
else:
    RL_data_in_exp = data_in_exp[::4,:10]
    RL_data_target_exp = data_target_exp[::4,:10]
    
    RL_output_exp = np.zeros(data_in_exp[::4,:10].shape)
    RL_TV_output_exp = np.zeros(data_in_exp[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_exp.shape)

### The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_exp = np.reshape(model_ConvNet.predict(data_in_exp.reshape([shape_exp[0]*shape_exp[1], shape_exp[2], shape_exp[3]]), 
                                                         verbose = 1), shape_exp)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_exp = Evaluate_metric(DAMN_model_output_exp, data_target_exp)

np.save("Data_files/Processed_data/Exp_DAMN_model_output.npy", DAMN_model_output_exp)
np.save("Data_files/Processed_data/Exp_DAMN_model_errors.npy", DAMN_model_errors_exp)

### Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the simulated data
start = time.time()
for i in range(RL_data_in_exp.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_exp[i])
    RL_output_exp[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_exp.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Exp_RL_output.npy", RL_output_exp)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_exp = Evaluate_metric(RL_output_exp, RL_data_target_exp)

np.save("Data_files/Processed_data/Exp_RL_errors.npy", RL_errors_exp)

### Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the simulated data
start = time.time()
for i in range(RL_data_in_exp.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_exp[i])
    RL_TV_output_exp[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_exp.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Exp_RLTV_output.npy", RL_TV_output_exp)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_exp = Evaluate_metric(RL_TV_output_exp, RL_data_target_exp)

np.save("Data_files/Processed_data/Exp_RLTV_errors.npy", RL_TV_errors_exp)